# Step 1b — ভুল জায়গার রেকর্ডিং খুঁজে বের করা (Colab, GPU)

**কী করে:** প্রতিটা রেকর্ডিং Whisper (large-v3-turbo, Bangla) দিয়ে লেখায় রূপান্তর করে, তারপর সেই লেখা `corpus.csv` এর ৯,৩২০টা বাক্যের (সিলেটি + বাংলা) সাথে মেলায়।
- **ok** — যে বাক্যের নম্বরে আছে, সেটাই বলা হয়েছে
- **moved** — অন্য একটা বাক্য স্পষ্টভাবে মিলছে (`suggested_row`)
- **shift** — পরপর কয়েকটা রেকর্ডিং একই পরিমাণ সরে আছে (যেমন সব ১ ঘর পিছিয়ে) → এগুলো সবচেয়ে নিশ্চিত
- **…_clash** — যে বাক্যে সরানো দরকার, সেটার ভালো রেকর্ডিং ওই speaker এর আগেই আছে (মানে এটা আসলে ডুপ্লিকেট টেক) → অটো ঠিক হবে না
- **uncertain** — পরিষ্কার মিল নেই, আবার অন্য বাক্যের সাথেও মেলে না → training এ থাকে, তবে শোনার তালিকায় যায়
- **check_ear** — Whisper কিছুই লিখতে পারেনি (ফাঁকা/নষ্ট অডিও) → training থেকে বাদ, শোনার তালিকায় যায়
- **moved/shift যেগুলো অটো ঠিক করা যায়নি** → ভুল লেবেল নিয়ে training এ ঢুকবে না, তাই `drop.csv` এ যায়

**ফলাফল (Drive এ `recheck/` ফোল্ডারে):** `report.csv` (সব ফাইল), `corrections.csv` (নিশ্চিত হয়ে অটো ঠিক করা হবে), `drop.csv` (training থেকে বাদ যাবে), `check_by_ear.csv` (শুনে দেখার তালিকা), `summary.csv`।
Data cleaning নোটবুক `corrections.csv` নিজে থেকে প্রয়োগ করে — আসল অডিও ফাইলে কোনো পরিবর্তন হয় না।

**মনে রাখবেন:** Whisper সিলেটি নিখুঁত লিখতে পারে না, তাই সীমা (threshold) সাবধানে রাখা হয়েছে — সন্দেহ হলে অটো ঠিক না করে শোনার তালিকায় দেয়।

**চালানোর নিয়ম**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. সময়: আনুমানিক ১–৩ ঘণ্টা। প্রতি ৫০০ ফাইল পরপর Drive এ সেভ হয়; disconnect হলে আবার Run all দিলে যেখানে থেমেছিল সেখান থেকে চলবে।
4. আগে Data cleaning নোটবুকের কনভার্সন (সেল ৪) হয়ে থাকলে WAV থেকে দ্রুত পড়বে; না হলে আসল ফাইল থেকে পড়বে (একটু ধীর)।

## 1. Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. সেটিংস

In [ ]:
ROOT = "/content/drive/MyDrive/Ultimate Thisis Data"   # folder with "Thesis Data 2", mapping.csv, corpus.csv
OUT  = "/content/wav_16k"                               # fast local Colab disk
SAVE = ROOT + "/wav_16k_tar"                            # finished speakers are kept here (survives disconnects)
SPEAKERS = ["A1", "A2", "A4", "A6", "A7"]   # speakers who withdrew consent were excluded
MODEL = "openai/whisper-large-v3-turbo"
BATCH = 16                      # lower to 8 if you get "CUDA out of memory"
STATUSES = ["primary", "second_voice", "alt_take", "unknown"]   # which files to check
RC = ROOT + "/recheck"
import os, pandas as pd, numpy as np, torch
os.makedirs(RC, exist_ok=True)
assert torch.cuda.is_available(), "No GPU - Runtime > Change runtime type > T4 GPU"
for f in ["Thesis Data 2", "mapping.csv", "corpus.csv"]:
    assert os.path.exists(f"{ROOT}/{f}"), f"'{f}' not found in ROOT"

## 3. লাইব্রেরি

In [ ]:
!pip -q install rapidfuzz

In [ ]:
%%writefile /content/recheck_lib.py
"""Find recordings whose audio does not match the sentence they are filed under.

Used by Step1b_Recording_Check.ipynb.  Pure logic (no model here) so it can be tested offline:
    match(transcripts, corpus)  -> per-clip scores, best candidate sentence and a verdict
Verdicts
    ok          audio matches the sentence it is filed under
    moved       audio clearly matches a different sentence (suggested_row)
    shift       part of a run of consecutive clips that are all moved by the same offset
    uncertain   neither the filed sentence nor any other one matches clearly
    check_ear   the transcript is too poor / empty to judge -> listen to it
"""
import re, unicodedata, numpy as np, pandas as pd
from rapidfuzz import fuzz, process

# Sylheti spelling uses ফ/খ/ছ where Standard Bangla uses প/ক/চ, and the ASR model writes Bangla.
# Folding similar sounds into one key makes Sylheti text, Bangla text and the ASR output comparable.
_PAIRS = {"খ": "ক", "ঘ": "গ", "ছ": "চ", "ঝ": "জ", "ঠ": "ট", "ঢ": "ড", "থ": "ত", "ধ": "দ",
          "ফ": "প", "ভ": "ব", "শ": "স", "ষ": "স", "ণ": "ন", "য": "জ",
          "ী": "ি", "ূ": "ু", "ৈ": "ই", "ৌ": "উ", "ঈ": "ই", "ঊ": "উ", "ঐ": "ই", "ঔ": "উ", "ং": "ঙ"}
FOLD = {ord(k): v for k, v in _PAIRS.items()}
FOLD.update({ord(c): None for c in "\u09cd\u0981\u0983\u09bc\u200d\u200c"})   # hasanta, chandrabindu, visarga, nukta, ZWJ/ZWNJ

def key(s) -> str:
    s = unicodedata.normalize("NFD", str(s or ""))
    s = s.replace("য\u09bc", "ই").replace("ড\u09bc", "র").replace("ঢ\u09bc", "র")   # য় ড় ঢ়
    s = s.translate(FOLD)
    s = re.sub(r"[^\u0980-\u09FF0-9a-zA-Z ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# ---- thresholds (scores are 0-100 similarity) ----
OK_SCORE, MOVE_MIN, MOVE_MARGIN, CLOSE, POOR = 55, 55, 15, 6, 25
STRONG_SCORE, STRONG_MARGIN, RUN_MIN = 65, 20, 3

def match(tr: pd.DataFrame, corpus: pd.DataFrame, chunk=2000, workers=-1) -> pd.DataFrame:
    """tr: wav_name, speaker, excel_row, order, asr   corpus: excel_row, sylheti, bangla"""
    tr = tr.copy().reset_index(drop=True)
    rows = corpus.excel_row.to_numpy()
    ks = [key(x) for x in corpus.sylheti]; kb = [key(x) for x in corpus.bangla]
    pos = {r: i for i, r in enumerate(rows)}
    q = [key(x) for x in tr.asr]
    tr["asr_key_len"] = [len(x) for x in q]
    # score against the filed sentence
    exp = np.zeros(len(tr))
    for i, (r, s) in enumerate(zip(tr.excel_row, q)):
        j = pos.get(int(r))
        exp[i] = 0 if j is None or not s else max(fuzz.ratio(s, ks[j]), fuzz.ratio(s, kb[j]))
    tr["score_filed"] = exp.round(1)
    # best sentence anywhere in the corpus
    best = np.zeros(len(tr)); brow = np.zeros(len(tr), dtype=int); second = np.zeros(len(tr))
    for a in range(0, len(tr), chunk):
        qq = q[a:a + chunk]
        m = np.maximum(process.cdist(qq, ks, scorer=fuzz.ratio, workers=workers, dtype=np.uint8),
                       process.cdist(qq, kb, scorer=fuzz.ratio, workers=workers, dtype=np.uint8))
        top = np.argsort(-m, axis=1)[:, :2]
        best[a:a + chunk] = m[np.arange(len(qq)), top[:, 0]]
        second[a:a + chunk] = m[np.arange(len(qq)), top[:, 1]]
        brow[a:a + chunk] = rows[top[:, 0]]
    tr["best_row"] = brow; tr["score_best"] = best; tr["score_second"] = second
    # identical sentences (same key) count as the same sentence
    same = [ks[pos[int(r)]] == ks[pos[int(b)]] if int(r) in pos else False for r, b in zip(tr.excel_row, brow)]
    v = np.full(len(tr), "uncertain", dtype=object)
    poor = (tr.asr_key_len < 4) | (best < POOR)
    ok = (exp >= OK_SCORE) | (exp >= best - CLOSE) | np.array(same)
    ok &= ~poor
    moved = ~ok & ~poor & (best >= MOVE_MIN) & (best - exp >= MOVE_MARGIN) & (best - second >= 3)
    v[ok] = "ok"; v[moved] = "moved"; v[poor] = "check_ear"
    tr["verdict"] = v
    tr["offset"] = np.where(moved, brow - tr.excel_row.astype(int), 0)
    # runs of consecutive clips (same speaker, recording order) moved by the same offset -> shift
    tr = tr.sort_values(["speaker", "order"]).reset_index(drop=True)
    run_id = ((tr.verdict != "moved") | (tr.offset != tr.offset.shift()) | (tr.speaker != tr.speaker.shift())).cumsum()
    size = tr.groupby(run_id).verdict.transform("size")
    tr.loc[(tr.verdict == "moved") & (size >= RUN_MIN), "verdict"] = "shift"
    tr["suggested_row"] = np.where(tr.verdict.isin(["moved", "shift"]), tr.best_row, tr.excel_row.astype(int))
    tr["auto_fix"] = (tr.verdict == "shift") | ((tr.verdict == "moved") & (tr.score_best >= STRONG_SCORE)
                                                  & (tr.score_best - tr.score_filed >= STRONG_MARGIN))
    # never auto-fix onto a sentence that this speaker already has a good recording of
    good = set(zip(tr.loc[tr.verdict == "ok", "speaker"], tr.loc[tr.verdict == "ok", "excel_row"].astype(int)))
    clash = [a and (s, r) in good for a, s, r in zip(tr.auto_fix, tr.speaker, tr.suggested_row)]
    tr.loc[clash, "auto_fix"] = False
    tr.loc[clash, "verdict"] = tr.loc[clash, "verdict"] + "_clash"
    # two clips of one speaker auto-fixed onto the same sentence -> keep neither automatic
    dup = tr[tr.auto_fix].duplicated(["speaker", "suggested_row"], keep=False)
    tr.loc[dup[dup].index, "auto_fix"] = False
    return tr

def summary(tr: pd.DataFrame) -> pd.DataFrame:
    t = pd.crosstab(tr.speaker, tr.verdict, margins=True, margins_name="total")
    t["auto_fix"] = tr.groupby("speaker").auto_fix.sum().reindex(t.index).fillna(tr.auto_fix.sum()).astype(int)
    return t

## 4. কোন ফাইলগুলো চেক হবে

In [ ]:
import os, tarfile
def restore_wavs():
    """Bring the converted WAVs back from Drive after a Colab reset (takes a few minutes)."""
    os.makedirs(OUT, exist_ok=True)
    for s in SPEAKERS:
        t = f"{SAVE}/{s}.tar"
        if os.path.exists(t) and not os.path.isdir(f"{OUT}/{s}"):
            with tarfile.open(t) as f: f.extractall(OUT)
            print("restored", s)

restore_wavs()
m = pd.read_csv(ROOT + "/mapping.csv", encoding="utf-8-sig")
m = m[m.status.isin(STATUSES)].reset_index(drop=True)
m["order"] = np.arange(len(m))
m["excel_row"] = m.excel_row.fillna(-1).astype(int)
m["id"] = m.original_file
corpus = pd.read_csv(ROOT + "/corpus.csv", encoding="utf-8-sig")
print(len(m), "files to check"); print(m.groupby(["speaker", "status"]).size().unstack(fill_value=0).to_string())

## 5. Whisper দিয়ে লেখা বের করা (সবচেয়ে বেশি সময় লাগে)

In [ ]:
import subprocess, soundfile as sf, threading
from concurrent.futures import ThreadPoolExecutor
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model=MODEL, torch_dtype=torch.float16, device=0)
GEN = {"language": "bengali", "task": "transcribe"}

def load(row):
    w = f"{OUT}/{row.wav_name}" if isinstance(row.wav_name, str) and row.wav_name else ""
    try:
        if w and os.path.exists(w):
            a, sr = sf.read(w, dtype="float32")
        else:
            src = f"{ROOT}/Thesis Data 2/" + row.original_file.replace("\\", "/")
            raw = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", "16000",
                                  "-f", "f32le", "-"], capture_output=True, timeout=120).stdout
            a = np.frombuffer(raw, dtype=np.float32)
        return a if len(a) > 1600 else None          # < 0.1 s = empty/broken
    except Exception:
        return None

for s in SPEAKERS:
    part = f"{RC}/transcripts_{s}.csv"
    done = pd.read_csv(part, encoding="utf-8-sig") if os.path.exists(part) else pd.DataFrame(columns=["id", "asr"])
    todo = m[(m.speaker == s) & ~m.id.isin(done.id)]
    print(f"{s}: {len(done)} done, {len(todo)} to go")
    for a in range(0, len(todo), 500):
        chunk = todo.iloc[a:a + 500]
        with ThreadPoolExecutor(8) as ex:
            audio = list(ex.map(load, chunk.itertuples()))
        ok = [i for i, x in enumerate(audio) if x is not None]
        texts = [""] * len(chunk)
        if ok:
            out = asr([{"raw": audio[i], "sampling_rate": 16000} for i in ok], batch_size=BATCH,
                      chunk_length_s=30, generate_kwargs=GEN)
            for i, o in zip(ok, out): texts[i] = o["text"].strip()
        done = pd.concat([done, pd.DataFrame({"id": chunk.id.values, "asr": texts})], ignore_index=True)
        done.to_csv(part + ".tmp", index=False, encoding="utf-8-sig"); os.replace(part + ".tmp", part)
        print(f"   {s}: {len(done)} saved")
print("All transcribed.")

## 6. মিলিয়ে দেখা ও রিপোর্ট

In [ ]:
import sys; sys.path.insert(0, "/content")
from recheck_lib import match, summary
tr = pd.concat([pd.read_csv(f"{RC}/transcripts_{s}.csv", encoding="utf-8-sig") for s in SPEAKERS
                if os.path.exists(f"{RC}/transcripts_{s}.csv")])
tr["asr"] = tr.asr.fillna("")
x = m.merge(tr, on="id", how="inner")
res = match(x[["wav_name", "speaker", "excel_row", "order", "asr", "status", "original_file"]], corpus)
txt = corpus.set_index("excel_row")
res["filed_sylheti"] = res.excel_row.map(txt.sylheti); res["suggested_sylheti"] = res.suggested_row.map(txt.sylheti)
cols = ["speaker", "status", "original_file", "wav_name", "excel_row", "verdict", "suggested_row", "auto_fix",
        "score_filed", "score_best", "asr", "filed_sylheti", "suggested_sylheti"]
res = res[cols].sort_values(["speaker", "excel_row"])
res.to_csv(RC + "/report.csv", index=False, encoding="utf-8-sig")
# only converted recordings (with a WAV name) can be re-labelled automatically
res[res.auto_fix & res.wav_name.notna()][["wav_name", "speaker", "excel_row", "suggested_row", "verdict",
    "score_filed", "score_best", "asr", "filed_sylheti", "suggested_sylheti"]].to_csv(
    RC + "/corrections.csv", index=False, encoding="utf-8-sig")
ear = res[(res.verdict != "ok") & ~res.auto_fix]
ear.to_csv(RC + "/check_by_ear.csv", index=False, encoding="utf-8-sig")
# clips that clearly say another sentence (but could not be moved safely) or are empty are left out of training
drop = ear[ear.wav_name.notna() & ear.verdict.str.startswith(("moved", "shift", "check_ear"))][["wav_name", "verdict", "asr"]]
drop_f = RC + "/drop.csv"
if os.path.exists(drop_f):                     # keep what you added by hand earlier
    drop = pd.concat([pd.read_csv(drop_f, encoding="utf-8-sig"), drop]).drop_duplicates("wav_name")
drop.to_csv(drop_f, index=False, encoding="utf-8-sig")
sm = summary(res); sm.to_csv(RC + "/summary.csv", encoding="utf-8-sig")
print(sm.to_string())
print(f"\nauto-fixed: {int(res.auto_fix.sum())}   to check by ear: {len(ear)}   left out of training (drop.csv): {len(drop)}")
print("Saved report.csv, corrections.csv, check_by_ear.csv, summary.csv in", RC)

## 7. নিজে শুনে যাচাই
নিচের সেল কিছু ফাইল বাজিয়ে শোনায় — অটো ঠিক করা গুলো থেকে কয়েকটা, আর শোনার তালিকা থেকে কয়েকটা। অটো-ফিক্সের নমুনা যদি বেশিরভাগ ঠিক থাকে, তাহলে `corrections.csv` এর উপর ভরসা করা যায়।

`recheck/drop.csv` এর ফাইলগুলো training থেকে বাদ যাবে। শুনে যদি দেখেন কোনোটা আসলে ঠিক আছে, `drop.csv` থেকে সেই লাইন মুছে দিন। আবার শোনার তালিকার কোনো `uncertain` ফাইল ভুল হলে তার `wav_name` `drop.csv` এ যোগ করুন।

In [ ]:
from IPython.display import Audio, display
def play(df, n=5):
    for _, r in df.sample(min(n, len(df))).iterrows():
        print(f"{r.speaker} {r.original_file}  [{r.verdict}]  filed row {r.excel_row} -> suggested {r.suggested_row}")
        print("   ASR      :", r.asr); print("   filed    :", r.filed_sylheti); print("   suggested:", r.suggested_sylheti)
        a = load(r) if hasattr(r, "original_file") else None
        if a is not None: display(Audio(a, rate=16000))
print("=== auto-fixed samples ==="); play(res[res.auto_fix])
print("\n=== check-by-ear samples ==="); play(ear)

---
**এরপর:** `Sylheti_Data_Cleaning.ipynb` আবার Run all করুন — `corrections.csv` প্রয়োগ হয়ে নতুন `metadata_splits.csv` আর `sylheti_asr/metadata.csv` তৈরি হবে।